<p><img style="float: right;" src="https://i.postimg.cc/YSFf8VV7/logo_PF_UPJS.png" alt="" width="100" height="85" /></p>
<p><strong>Authors:</strong> Dominik Borovský<sup>1</sup> & Jozef Hanč<sup>2</sup></p>
<p><em>2026, Faculty of Science, Pavol Jozef Šafárik University in Košice, Slovakia</em></p>
<p><sup>1</sup><a href="mailto:dominik.borovsky@student.upjs.sk">dominik.borovsky@student.upjs.sk,</a> <sup>2</sup><a href="mailto:jozef.hanc@upjs.sk">jozef.hanc@upjs.sk</a> </p>



# **Chapter 3: Pictures of Trajectories**




In this chapter you will learn to plot functions and data in SageMath: single plots, families of curves, and how to combine and decorate them. A plot is not decoration; it is how you *see* an equation.



## **3.1 The task**

> The goalkeeper wants to see, in one picture, where the ball can and cannot land, for every kick angle at once. Draw it.



## **3.2 The trajectory as a curve**

In Chapter 2 the ball's position was given by two functions of time: the distance $x(t) = v_0\cos\alpha \, t$, growing evenly for as long as the ball flies, and the height $y(t) = v_0\sin\alpha \, t - \tfrac{1}{2}g t^2$, rising and then falling. For a picture we want height $y$ against distance $x$, $y$ as a function of $x$: drop the clock and keep the curve. The procedure is: solve the distance equation for the time $t$, and substitute that time into the height $y(t)$. Both steps are exactly what `solve()` and substitution were for in Chapter 2, so we do not do them by hand - we rebuild the two flight functions and let SageMath eliminate $t$ itself.

One naming decision comes first. The equation to solve is *distance $= x$*, with $x$ a plain unknown - but the letter $x$ is already taken by the distance function, and `solve(x(t) == x, t)` ends in an error. For SageMath the function and the unknown are not the same thing, and they are not the same thing in mathematics either: $x(t)$ is a rule, $x$ is the number we are asking for. The trick for such collisions is a trailing underscore: the functions are called `x_` and `y_`, the unknown stays `x`. In code the underscore keeps the two apart; `show()` drops it when typesetting, and both display as plain $x$.



In [ ]:
var('t alpha v0 g x')

# flight functions: underscore in the code, plain x and y in show()
x_(t) = v0*cos(alpha)*t
y_(t) = v0*sin(alpha)*t - 1/2*g*t^2

# solve "distance = x" for the time, keep the (only) solution
t_hit = solve(x_(t) == x, t)[0].rhs()
show(t_hit)


In [ ]:
# substitute the time into the height: y as a function of x
y_traj(x) = y_(t).subs(t = t_hit).trig_simplify()
show(y_traj(x))




A parabola, as promised by every physics teacher. The simplest plot in SageMath is `plot()`: its arguments are what to plot and the range of the horizontal axis, written as `xmin` and `xmax`. The names say what they do - the same style as the `ymin` and `ymax` we will meet below.




In [ ]:
# a single 45 degree kick at 15 m/s: substitute numbers first
kappa45(x) = y_traj(x).subs(alpha = pi/4, v0 = 15, g = 9.81)
plot(kappa45(x), xmin = 0, xmax = 25)




The axes scale automatically, and the curve is there: up, over, down.

> **Note:** `plot()` needs to know *where to evaluate*, so the interval is not optional. If SageMath complains about not knowing what to plot, it almost always means you forgot `xmin = ...` or `xmax = ...`.




## **3.3 All the angles at once**

One curve is one angle. The goalkeeper wants the whole family: angles from 20 to 70 degrees. The tool for repeating a computation with different values is the list comprehension, `[expression for item in list]`, met here for the first time. It works for plots too, because in SageMath a plot is an object like any number:



In [ ]:
angles = [20, 30, 40, 50, 60, 70]      # in degrees

# one plot object per angle, collected in a list
plots = [plot(y_traj(x).subs(alpha = a*pi/180, v0 = 15, g = 9.81),
              xmin = 0, xmax = 25,
              color = hue(a/90),
              legend_label = str(a) + ' degrees')
         for a in angles]




Each plot got a `color` and a `legend_label` so we can tell them apart. Plots are combined by simply adding them, and the sum is drawn with `show()`. A habit worth forming right here: *build the sum, give it a name, and only then show it*. The sum is a plot object like any other, so it deserves a name like any other; the next sections will keep adding to this picture, and nobody wants to retype a list comprehension for that.




In [ ]:
# the + operator overlays plots; store the sum under a name
family_plot = sum(plots)

show(family_plot, ymin=0)




Six parabolas with a clear pattern: 20 and 70 degrees land together, 30 and 60 together, and 40 with 50 almost together, exactly the $\sin 2\alpha$ symmetry from Chapter 2. The tallest arc lands no farther than the flattest one.




## **3.4 The envelope: where no ball lands**

Look at the family of curves and you can see a ceiling: an outer border the balls touch but never cross. It is called the *envelope*, and SageMath can compute it, not just draw it. The trick is a substitution that makes the algebra pleasant. Write $u = \tan\alpha$, and use $1/\cos^2\alpha = 1 + \tan^2\alpha$:



In [ ]:
var('u')

# the same trajectory, but in terms of u = tan(alpha)
y_u(x) = u*x - g*x^2/(2*v0^2)*(1 + u^2)
show(y_u(x))




For a fixed distance $x$, the highest kick is the best $u$. The maximum of $y_u$ over $u$ sits where its derivative in $u$ is zero:




In [ ]:
u_best = solve(diff(y_u(x), u) == 0, u)[0].rhs()
show(u_best)




Substitute this optimal kick back into the trajectory and the envelope falls out:




In [ ]:
envelope(x) = y_u(x).subs(u = u_best)
show(envelope(x))




$$y_{env} = \dfrac{v_0^2}{2g} - \dfrac{g\,x^2}{2v_0^2}$$

A parabola again, but an upside-down one, sitting on top of all the others. Each point of it belongs to one ball, the ball kicked at just the right angle to be there.

Now draw everything together. Plots take a `linestyle` option, `'--'` for a dashed curve:




In [ ]:
env_plot = plot(envelope(x).subs(v0 = 15, g = 9.81), xmin = 0, xmax = 25,
                color = 'black', linestyle = '--', legend_label = 'envelope')

# the family from before plus the envelope: one named picture
combined_plot = family_plot + env_plot

show(combined_plot, figsize = 5, ymin = 0)




There is the goalkeeper's picture: everything inside the dashed parabola can be hit by some angle, everything above it is safe. The apex of the envelope, $v_0^2/2g$ at $x = 0$, is the highest a 15 m/s ball can ever reach, straight up.




## **3.5 Making the picture readable**

A plot you cannot read is a wasted computation. Here is the payoff of the naming habit: the same `combined_plot` from the last section, dressed better, without rebuilding a single curve. The options are the everyday ones:



In [ ]:
show(combined_plot,
     figsize = 6,             # size of the figure
     ymin = 0, ymax = 12,     # vertical range
     frame = True,            # box around the picture instead of axes
     axes_labels = ['$x$ [m]', '$y$ [m]'],   # axis captions
     gridlines = True)        # background grid




Three details are worth a habit. First, `axes_labels` accept LaTeX: write the caption between dollar signs, and the `$x$ [m]` on the axis comes out typeset in mathematical script, the italic $x$ you know from formulas. Second, `axes_labels` is where units belong, per our convention from Chapter 1: numbers in the notebook stay bare, units live in labels. Third, `ymin = 0` is not cosmetic: without it, SageMath happily shows the ball landing below ground level.

> **Note:** Colors can be given as `hue(number)` between 0 and 1, which spreads a family of curves evenly over the rainbow, or as names like `'red'`, `'black'`. The `legend_label` is plain text, so it is built with `str(a) + ' degrees'` when the label has to contain a number.

Curves say "somewhere up there"; a dot says "exactly here". Worth marking are the *apexes* of the kicks, the points where each ball stops rising. Their time we know from Chapter 2: half the flight time, $t_{\mathrm{apex}} = v_0\sin\alpha / g$. For a picture we need the *place* of each apex, the pair (distance, height). The cleanest tool for that is a `lambda`. It is part of Python, not of SageMath's mathematics: with it, one easily writes down short procedures that accept one or more inputs, process them and provide output in any form - here, one input (the angle in degrees) and one number out, with our kick (`v0 = 15`, `g = 9.81`) and the degree-to-radian conversion baked inside:



In [ ]:
# the apex time, from the Chapter 2 symmetry
t_apex = v0*sin(alpha)/g

# a short procedure per coordinate: angle in degrees in, place in meters out
x_apex = lambda a: N(x_(t_apex).subs(alpha = a*pi/180, v0 = 15, g = 9.81))
y_apex = lambda a: N(y_(t_apex).subs(alpha = a*pi/180, v0 = 15, g = 9.81))

show(x_apex(45))
show(y_apex(45))




Read a definition from the left: the keyword `lambda`, the name of the input (`a`, the angle in degrees), a colon, and what to compute. A lambda is *not* a mathematical function like `x_(t) = ...`: it holds no equation to solve, nothing to differentiate, no formula to display - it is pure computing. Above, the 45 degree kick puts its apex 11.5 m out and 5.7 m up.

> **Note:** If you already know some Python, you may have met `def` statements: `def f(a): return ...` works the same way, with the body written on one or more indented lines. A `lambda` is the same idea squeezed into a single expression - the right tool when the procedure fits on one line, a `def` when it does not.

The payoff comes in the list below: with all the substituting hidden inside the procedures, it stays as flat as the list of parabolas in Section 3.3. And this time we take the angles further, up to 90 degrees:



In [ ]:
# one (distance, height) pair per angle
# the angles from before, plus the straight-up kick
apex_angles = angles + [80, 90]

apexes = [(x_apex(a), y_apex(a)) for a in apex_angles]

show(apexes)




Eight pairs, from `(7.4, 1.3)` at 20 degrees up to `(0, 11.5)` at 90 degrees. The 90 degree kick has no parabola in the family above: kicked straight up, the ball gains no distance, and asking `y_traj` for that angle would mean dividing by $\cos 90^\circ = 0$. Its apex is still a perfectly good point - no distance, all height. Such a list is drawn with `point()`, which wants exactly this shape - a list of $(x, y)$ pairs - and nothing else:




In [ ]:
# one dot per apex, on top of the picture
apex_plot = point(apexes, size = 40, color = 'red')

show(combined_plot + apex_plot, figsize = 5, ymin = 0)




Each dot sits on top of its own parabola, and the 90 degree dot sits exactly on the envelope's top, $v_0^2/2g$: the highest point of the picture belongs to the ball kicked straight up.

Dots mark places; a line tells the story along them. `list_plot()` takes the very same list of pairs and adds one parameter worth knowing: `plotjoined = True` runs a line through the dots:



In [ ]:
# the same pairs, joined into one line
apex_line = list_plot(apexes, plotjoined = True, color = 'red')

show(combined_plot + apex_plot + apex_line, figsize = 5, ymin = 0)




The line makes the pattern visible: the apexes climb a smooth arc that ends on the envelope's top. That arc is half of an ellipse - the apex of every kick angle lies on it, and its highest point is the ceiling itself.

> **Note:** `point()` for a handful of marked dots; `list_plot()` with `plotjoined = True` when the dots belong to one story.



## **Exercises**

**Exercise 1.** Rebuild the angle family for `v0 = 20` and `v0 = 10`, each with its own envelope. How does the envelope change? Check your reading against the formula $v_0^2/(2g)$ for its apex.

**Exercise 2.** A fence of height 2 m stands 18 m away. Plot the 45 degree trajectory together with a horizontal line at 2 m (hint: `plot(2, x, xmin = 0, xmax = 25)` plots a constant; the plain `x` stays, because a bare constant tells SageMath nothing about the variable) and decide by eye whether the kick clears it. Then confirm with `solve()` from Chapter 2.

**Exercise 3.** Plot the trajectories of a ball kicked from the edge of a 10 m cliff, angles 0 to 60 degrees. Where does the landing symmetry from Chapter 2 (the $\sin 2\alpha$ pairs) break, and why?

**Exercise 4.** The flight *time* also depends on the angle. Plot `t_land` from Chapter 2 as a function of the angle, from 0 to 90 degrees, with `axes_labels`. Mark the angle where the time is longest and explain in one sentence why it is not 45 degrees.



## **3.6 What you can do now**

- **`plot(expr, xmin, xmax)`** draws a function of one variable; the axis range is required, and its names match `ymin` and `ymax`.
- **A trailing underscore** (`x_`, `y_`) lets a function and a symbol share a letter; `show()` typesets both as plain $x$.
- **Plot objects can be stored in lists** (list comprehension) and **added together** with `+` to overlay families of curves; give the sum a name (`family_plot`, `combined_plot`) before `show()`-ing it, so it can be reused and extended.
- **`point(pairs)`** marks a list of $(x, y)$ points on a picture, here the apexes of the trajectories; **`list_plot(pairs, plotjoined = True)`** joins the dots into a line.
- **`lambda a: ...`** is a one-line procedure from Python - inputs in, processed output out, in any form; it is not a mathematical function, so nothing to solve or display. A `def` statement works the same way for multiline procedures.
- **`color`, `linestyle`, `legend_label`, `axes_labels`, `figsize`, `gridlines`, `ymin`, `frame`** are the everyday formatting options; `axes_labels` accept LaTeX between dollar signs.
- **The envelope of a family of curves** is found by maximizing over the family parameter, here with a substitution and `diff()` + `solve()`: a symbolic result, drawn only at the end.
- **Units live in axis labels**, not inside the numbers, following the convention from Chapter 1.




In [ ]:
# empty cells for exercises/experiments - feel free to add new cells anywhere by clicking "Add Bellow" button
